# Air tracker data pipeline

This notebook fetches airport, flight, aircraft, and delay data from AeroDataBox and stores it in MySQL. API calls are paced, cached, and capped per run to stay within RapidAPI rate limits.

In [7]:
import hashlib
import json
import os
import random
import statistics
import time
from datetime import datetime, timedelta
from email.utils import parsedate_to_datetime
from getpass import getpass
from pathlib import Path

import requests

from Flight_analytics_database import get_database_config, make_flight_id, to_local_datetime

API_KEY = (os.getenv("RAPIDAPI_KEY") or getpass("RapidAPI key: ")).strip()
if not API_KEY:
    raise ValueError("A RapidAPI key is required. Set RAPIDAPI_KEY or enter it at the prompt.")
API_HOST = "aerodatabox.p.rapidapi.com"
HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST,
}
REQUEST_INTERVAL_SECONDS = 1.1  # Stay below a one-request-per-second plan.
MAX_RATE_LIMIT_WAIT_SECONDS = 120
MAX_API_REQUESTS_PER_RUN = int(os.getenv("MAX_API_REQUESTS_PER_RUN", "60"))
MAX_AIRCRAFT_REQUESTS_PER_RUN = int(os.getenv("MAX_AIRCRAFT_REQUESTS_PER_RUN", "15"))
ENABLE_AIRCRAFT_ENRICHMENT = os.getenv("ENABLE_AIRCRAFT_ENRICHMENT", "false").lower() == "true"
STANDARD_CACHE_TTL_SECONDS = 7 * 24 * 60 * 60
API_CACHE_DIR = Path(".cache") / "aerodatabox"
API_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DB_CONFIG = get_database_config(prompt=True)
AIRPORT_CODES = [
    "DXB", "SIN", "LHR", "JFK", "SYD", "DOH", "BKK", "CDG",
    "MAA", "DEL", "BOM", "BLR", "HYD", "CCU", "COK",
]
AIRPORT_ICAO = {
    "DXB": "OMDB", "SIN": "WSSS", "LHR": "EGLL", "JFK": "KJFK",
    "SYD": "YSSY", "DOH": "OTHH", "BKK": "VTBS", "CDG": "LFPG",
    "MAA": "VOMM", "DEL": "VIDP", "BOM": "VABB", "BLR": "VOBL",
    "HYD": "VOHS", "CCU": "VECC", "COK": "VOCI",
}

print("Configuration ready")

Configuration ready


In [8]:
import mysql.connector


def get_connection():
    return mysql.connector.connect(**DB_CONFIG)


_session = requests.Session()
_session.headers.update(HEADERS)
_last_request_at = 0.0
_requests_this_run = 0
_api_paused = False
_NO_CONTENT_MARKER = {"__aerodatabox_no_content__": True}


def _cache_path(url, params):
    cache_key = json.dumps([url, params or {}], sort_keys=True, separators=(",", ":"))
    digest = hashlib.sha256(cache_key.encode("utf-8")).hexdigest()
    return API_CACHE_DIR / f"{digest}.json"


def _read_cache(path, ttl_seconds):
    if not ttl_seconds or not path.exists():
        return None
    if time.time() - path.stat().st_mtime > ttl_seconds:
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def _reset_seconds(value):
    if not value:
        return None
    try:
        numeric = float(value)
        return max(0.0, numeric - time.time()) if numeric > time.time() else max(0.0, numeric)
    except (TypeError, ValueError):
        try:
            return max(0.0, parsedate_to_datetime(value).timestamp() - time.time())
        except (TypeError, ValueError, OverflowError):
            return None


def _remaining_quota(headers):
    for name in (
        "x-rate-limit-rapid-free-plans-hard-limit-remaining",
        "x-ratelimit-requests-remaining",
    ):
        value = headers.get(name)
        if value is not None:
            try:
                return int(float(value))
            except ValueError:
                pass
    return None


def _quota_summary(headers):
    values = []
    for name, value in headers.items():
        lowered = name.lower()
        if ("ratelimit" in lowered or "rate-limit" in lowered) and ("remaining" in lowered or "reset" in lowered):
            values.append(f"{name}={value}")
    return ", ".join(values) or "no quota headers returned"


def _response_message(response):
    try:
        payload = response.json()
        if isinstance(payload, dict):
            return str(payload.get("message") or payload.get("error") or payload)[:300]
        return str(payload)[:300]
    except ValueError:
        return (response.text or response.reason or "No response body")[:300]


def _write_cache(path, data):
    try:
        path.write_text(json.dumps(data), encoding="utf-8")
    except OSError as error:
        print(f"Warning: could not update API cache: {error}")


def request_json(url, params=None, retries=3, cache_ttl=0):
    global _api_paused, _last_request_at, _requests_this_run

    cache_file = _cache_path(url, params)
    cached = _read_cache(cache_file, cache_ttl)
    if cached == _NO_CONTENT_MARKER:
        return None
    if cached is not None:
        return cached
    if _api_paused:
        return None
    if _requests_this_run >= MAX_API_REQUESTS_PER_RUN:
        _api_paused = True
        print(f"API request budget reached ({MAX_API_REQUESTS_PER_RUN}); remaining work deferred.")
        return None

    for attempt in range(retries):
        if _requests_this_run >= MAX_API_REQUESTS_PER_RUN:
            _api_paused = True
            print(f"API request budget reached ({MAX_API_REQUESTS_PER_RUN}); remaining work deferred.")
            return None
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            response = _session.get(url, params=params, timeout=30)
            _last_request_at = time.monotonic()
            _requests_this_run += 1

            if response.status_code == 429:
                quota_remaining = _remaining_quota(response.headers)
                quota_reset = _reset_seconds(response.headers.get("x-ratelimit-requests-reset"))
                retry_after = _reset_seconds(
                    response.headers.get("Retry-After")
                    or response.headers.get("x-ratelimit-reset")
                )
                if quota_remaining == 0:
                    _api_paused = True
                    reset_note = f" Reset is in about {quota_reset / 3600:.1f} hours." if quota_reset else ""
                    print("RapidAPI quota is exhausted; API calls are paused for this run." + reset_note)
                    return None
                if retry_after is None or retry_after > MAX_RATE_LIMIT_WAIT_SECONDS:
                    _api_paused = True
                    reset_note = f" Try again in about {retry_after / 60:.1f} minutes." if retry_after else ""
                    print("The rate-limit window is too long to wait in the notebook; calls are paused." + reset_note)
                    return None
                if attempt < retries - 1:
                    wait_seconds = max(1.0, retry_after) + random.uniform(0.1, 0.5)
                    print(f"Rate limited; retrying in {wait_seconds:.1f} seconds.")
                    time.sleep(wait_seconds)
                    continue
                print("Rate limit still active after retries; remaining work deferred.")
                return None

            if response.status_code == 204:
                _write_cache(cache_file, _NO_CONTENT_MARKER)
                print(f"API 204 (no data): {response.url}")
                return None

            if 400 <= response.status_code < 500:
                print(
                    f"API rejected request ({response.status_code}): {_response_message(response)}\n"
                    f"URL: {response.url}\nQuota: {_quota_summary(response.headers)}"
                )
                return None  # Retrying an unchanged client error only wastes quota.

            response.raise_for_status()
            try:
                data = response.json()
            except ValueError:
                print(f"API returned non-JSON data ({response.status_code}): {response.url}")
                return None
            _write_cache(cache_file, data)
            remaining = _remaining_quota(response.headers)
            print(f"API {response.status_code}: {response.url} | {_quota_summary(response.headers)}")
            if remaining is not None and remaining <= 10:
                print(f"Warning: only {remaining} API quota requests remain.")
            if remaining == 0:
                _api_paused = True
            return data
        except (requests.RequestException, ValueError) as error:
            if attempt == retries - 1:
                print(f"Request failed: {error}")
                return None
            time.sleep(3 * (attempt + 1) + random.uniform(0.1, 0.5))
    return None


def execute_write(sql, rows):
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.executemany(sql, rows)
        connection.commit()
        return cursor.rowcount
    except mysql.connector.Error:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


def fetch_flights(iata_code, flight_date):
    # Date-range values are path parameters. Two 12-hour requests work on plans
    # whose FIDS window is limited to 12 hours.
    windows = [("00:00", "11:59"), ("12:00", "23:59")]
    combined = {"departures": [], "arrivals": []}
    received_data = False
    params = {
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true",
        "withCodeshared": "false",
        "withCargo": "false",
        "withPrivate": "false",
    }
    for start_time, end_time in windows:
        from_local = f"{flight_date}T{start_time}"
        to_local = f"{flight_date}T{end_time}"
        url = f"https://{API_HOST}/flights/airports/iata/{iata_code}/{from_local}/{to_local}"
        data = request_json(url, params=params, cache_ttl=24 * 60 * 60)
        if not isinstance(data, dict):
            continue
        received_data = True
        for direction in ("departures", "arrivals"):
            records = data.get(direction)
            if isinstance(records, list):
                combined[direction].extend(records)
    return combined if received_data else None


def check_api_connection(icao_code="VOMM"):
    """Validate the key through AeroDataBox's FREE TIER health endpoint."""
    result = request_json(
        f"https://{API_HOST}/health/services/airports/{icao_code}/feeds",
        retries=1,
    )
    print("API authentication and connectivity are working." if isinstance(result, dict) else "API check failed; see the response above.")
    return result


print("Database and API helpers ready. Run check_api_connection() for a zero-unit diagnostic call.")

Database and API helpers ready. Run check_api_connection() for a zero-unit diagnostic call.


In [9]:
airport_rows = []
for iata_code in AIRPORT_CODES:
    data = request_json(
        f"https://{API_HOST}/airports/iata/{iata_code}",
        cache_ttl=STANDARD_CACHE_TTL_SECONDS,
    )
    if not isinstance(data, dict):
        continue
    airport_rows.append((
        data.get("icao"),
        data.get("iata"),
        data.get("fullName"),
        data.get("municipalityName"),
        (data.get("country") or {}).get("name"),
        (data.get("continent") or {}).get("name"),
        (data.get("location") or {}).get("lat"),
        (data.get("location") or {}).get("lon"),
        data.get("timeZone"),
    ))

airport_sql = """
    INSERT INTO airport
    (icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        name = VALUES(name), city = VALUES(city), country = VALUES(country),
        continent = VALUES(continent), latitude = VALUES(latitude),
        longitude = VALUES(longitude), timezone = VALUES(timezone)
"""

if airport_rows:
    execute_write(airport_sql, airport_rows)
print(f"Upserted {len(airport_rows)} airports")

API 200: https://aerodatabox.p.rapidapi.com/airports/iata/DXB | X-RateLimit-rapid-free-plans-hard-limit-Remaining=499981, X-RateLimit-rapid-free-plans-hard-limit-Reset=2590214, X-RateLimit-API-Units-Remaining=378, X-RateLimit-API-Units-Reset=2590214, X-RateLimit-Requests-Remaining=1581, X-RateLimit-Requests-Reset=2590214
API 200: https://aerodatabox.p.rapidapi.com/airports/iata/SIN | X-RateLimit-Requests-Remaining=1580, X-RateLimit-Requests-Reset=2590213, X-RateLimit-API-Units-Remaining=377, X-RateLimit-API-Units-Reset=2590213, X-RateLimit-rapid-free-plans-hard-limit-Remaining=499980, X-RateLimit-rapid-free-plans-hard-limit-Reset=2590213
API 200: https://aerodatabox.p.rapidapi.com/airports/iata/LHR | X-RateLimit-API-Units-Remaining=376, X-RateLimit-API-Units-Reset=2590211, X-RateLimit-Requests-Remaining=1579, X-RateLimit-Requests-Reset=2590211, X-RateLimit-rapid-free-plans-hard-limit-Remaining=499979, X-RateLimit-rapid-free-plans-hard-limit-Reset=2590211
API 200: https://aerodatabox.p.

In [10]:
flight_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
flight_rows_by_id = {}

for airport_iata in AIRPORT_CODES:
    data = fetch_flights(airport_iata, flight_date)
    if not isinstance(data, dict):
        continue
    departures = data.get("departures")
    arrivals = data.get("arrivals")
    if not isinstance(departures, list):
        departures = []
    if not isinstance(arrivals, list):
        arrivals = []
    records = departures + arrivals
    for flight in records:
        if not isinstance(flight, dict):
            continue
        departure = flight.get("departure") or {}
        arrival = flight.get("arrival") or {}
        aircraft = flight.get("aircraft") or {}
        airline = flight.get("airline") or {}
        departure_airport = departure.get("airport") or {}
        arrival_airport = arrival.get("airport") or {}
        scheduled_departure = to_local_datetime((departure.get("scheduledTime") or {}).get("local"))
        actual_departure = to_local_datetime((departure.get("actualTime") or {}).get("local"))
        revised_departure = to_local_datetime((departure.get("revisedTime") or {}).get("local"))
        scheduled_arrival = to_local_datetime((arrival.get("scheduledTime") or {}).get("local"))
        actual_arrival = to_local_datetime((arrival.get("actualTime") or {}).get("local"))
        revised_arrival = to_local_datetime((arrival.get("revisedTime") or {}).get("local"))
        flight_number = flight.get("number")
        if not flight_number:
            continue
        origin_iata = departure_airport.get("iata") or airport_iata
        destination_iata = arrival_airport.get("iata")
        flight_id = make_flight_id(
            flight_number, origin_iata, destination_iata,
            scheduled_departure, scheduled_arrival, flight_date,
        )
        flight_row = (
            flight_id,
            flight_number,
            aircraft.get("reg"),
            origin_iata,
            destination_iata,
            scheduled_departure,
            actual_departure,
            revised_departure,
            scheduled_arrival,
            actual_arrival,
            revised_arrival,
            flight.get("status"),
            airline.get("iata"),
            aircraft.get("model"),
        )
        existing = flight_rows_by_id.get(flight_id)
        if existing is None or sum(value is not None for value in flight_row) > sum(value is not None for value in existing):
            flight_rows_by_id[flight_id] = flight_row

flight_rows = list(flight_rows_by_id.values())

flight_sql = """
    INSERT INTO flights
    (flight_id, flight_number, aircraft_registration, origin_iata, destination_iata,
     scheduled_departure, actual_departure, revised_departure,
     scheduled_arrival, actual_arrival, revised_arrival,
     status, airline_code, aircraft_model)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        flight_number = VALUES(flight_number), aircraft_registration = VALUES(aircraft_registration),
        origin_iata = VALUES(origin_iata), destination_iata = VALUES(destination_iata),
        scheduled_departure = VALUES(scheduled_departure), actual_departure = VALUES(actual_departure),
        revised_departure = VALUES(revised_departure),
        scheduled_arrival = VALUES(scheduled_arrival), actual_arrival = VALUES(actual_arrival),
        revised_arrival = VALUES(revised_arrival),
        status = VALUES(status), airline_code = VALUES(airline_code), aircraft_model = VALUES(aircraft_model)
"""

if flight_rows:
    execute_write(flight_sql, flight_rows)
print(f"Upserted {len(flight_rows)} flights for {flight_date}")

API 200: https://aerodatabox.p.rapidapi.com/flights/airports/iata/DXB/2026-09-26T00:00/2026-09-26T11:59?withLeg=true&direction=Both&withCancelled=true&withCodeshared=false&withCargo=false&withPrivate=false | X-RateLimit-Requests-Remaining=1566, X-RateLimit-Requests-Reset=2590192, X-RateLimit-rapid-free-plans-hard-limit-Remaining=499966, X-RateLimit-rapid-free-plans-hard-limit-Reset=2590192, X-RateLimit-API-Units-Remaining=362, X-RateLimit-API-Units-Reset=2590192
API 200: https://aerodatabox.p.rapidapi.com/flights/airports/iata/DXB/2026-09-26T12:00/2026-09-26T23:59?withLeg=true&direction=Both&withCancelled=true&withCodeshared=false&withCargo=false&withPrivate=false | X-RateLimit-rapid-free-plans-hard-limit-Remaining=499965, X-RateLimit-rapid-free-plans-hard-limit-Reset=2590190, X-RateLimit-API-Units-Remaining=360, X-RateLimit-API-Units-Reset=2590190, X-RateLimit-Requests-Remaining=1565, X-RateLimit-Requests-Reset=2590190
API 200: https://aerodatabox.p.rapidapi.com/flights/airports/iata/

In [11]:
def fetch_aircraft(registration):
    return request_json(
        f"https://{API_HOST}/aircrafts/reg/{registration}",
        cache_ttl=STANDARD_CACHE_TTL_SECONDS,
    )


# Flight responses already contain registration and model. Populate those fields
# locally so one extra API call per aircraft is not required.
connection = get_connection()
cursor = connection.cursor()
cursor.execute("""
    INSERT INTO aircraft (registration, model)
    SELECT aircraft_registration, MAX(aircraft_model)
    FROM flights
    WHERE aircraft_registration IS NOT NULL AND aircraft_registration <> ''
    GROUP BY aircraft_registration
    ON DUPLICATE KEY UPDATE model = COALESCE(VALUES(model), model)
""")
connection.commit()
cursor.close()
connection.close()

registrations = []
if ENABLE_AIRCRAFT_ENRICHMENT:
    connection = get_connection()
    cursor = connection.cursor(dictionary=True)
    cursor.execute("""
        SELECT registration AS aircraft_registration
        FROM aircraft
        WHERE manufacturer IS NULL AND icao_type_code IS NULL AND owner IS NULL
        ORDER BY registration
        LIMIT %s
    """, (MAX_AIRCRAFT_REQUESTS_PER_RUN,))
    registrations = [row["aircraft_registration"] for row in cursor.fetchall()]
    cursor.close()
    connection.close()
else:
    print("Aircraft API enrichment disabled; using registration/model from flight data.")

aircraft_rows = []
for registration in registrations:
    record = fetch_aircraft(registration)
    if not isinstance(record, dict):
        continue
    type_name = record.get("typeName") or record.get("model")
    aircraft_rows.append((
        record.get("reg") or record.get("registration") or registration,
        record.get("model") or type_name,
        type_name.split()[0] if isinstance(type_name, str) else None,
        record.get("icaoCode") or record.get("typeCode"),
        record.get("airlineName") or record.get("owner"),
    ))

aircraft_sql = """
    INSERT INTO aircraft (registration, model, manufacturer, icao_type_code, owner)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        model = VALUES(model), manufacturer = VALUES(manufacturer),
        icao_type_code = VALUES(icao_type_code), owner = VALUES(owner)
"""
if aircraft_rows:
    execute_write(aircraft_sql, aircraft_rows)
print(f"API-enriched {len(aircraft_rows)} aircraft")


# Calculate airport delays from the flight timestamps already downloaded. This
# replaces 15 Tier-3 API requests (six API units each).
delay_rows = []
connection = get_connection()
cursor = connection.cursor(dictionary=True)
for iata_code in AIRPORT_CODES:
    cursor.execute("""
        SELECT status,
               CASE
                   WHEN origin_iata = %s AND actual_departure IS NOT NULL
                       THEN TIMESTAMPDIFF(MINUTE, scheduled_departure, actual_departure)
                   WHEN destination_iata = %s AND actual_arrival IS NOT NULL
                       THEN TIMESTAMPDIFF(MINUTE, scheduled_arrival, actual_arrival)
               END AS delay_minutes
        FROM flights
        WHERE (origin_iata = %s AND DATE(scheduled_departure) = %s)
           OR (destination_iata = %s AND DATE(scheduled_arrival) = %s)
    """, (iata_code, iata_code, iata_code, flight_date, iata_code, flight_date))
    records = cursor.fetchall()
    delay_minutes = [max(0, row["delay_minutes"]) for row in records if row["delay_minutes"] is not None]
    delayed_flights = sum(
        row["status"] == "Delayed" or (row["delay_minutes"] or 0) > 15
        for row in records
    )
    delay_rows.append({
        "airport_iata": iata_code,
        "delay_date": flight_date,
        "total_flights": len(records),
        "delayed_flights": delayed_flights,
        "avg_delay_min": round(statistics.mean(delay_minutes)) if delay_minutes else 0,
        "median_delay_min": round(statistics.median(delay_minutes)) if delay_minutes else 0,
        "canceled_flights": sum(row["status"] in ("Canceled", "Cancelled") for row in records),
    })
cursor.close()
connection.close()

delay_sql = """
    INSERT INTO airport_delays
    (airport_iata, delay_date, total_flights, delayed_flights, avg_delay_min,
     median_delay_min, canceled_flights)
    VALUES (%(airport_iata)s, %(delay_date)s, %(total_flights)s, %(delayed_flights)s,
            %(avg_delay_min)s, %(median_delay_min)s, %(canceled_flights)s)
"""
connection = get_connection()
cursor = connection.cursor()
try:
    cursor.executemany(
        "DELETE FROM airport_delays WHERE airport_iata = %(airport_iata)s AND delay_date = %(delay_date)s",
        delay_rows,
    )
    cursor.executemany(delay_sql, delay_rows)
    connection.commit()
finally:
    cursor.close()
    connection.close()
print(f"Upserted delay summaries for {len(delay_rows)} airports")

Aircraft API enrichment disabled; using registration/model from flight data.
API-enriched 0 aircraft
Upserted delay summaries for 15 airports
